# Fase 3.5 — Re-verificacion de conflictos (criterio v2)

Rework del 2026-07-10: al revisar `/hallazgos` en la app, se detecto que la mayoria de los 84
`possible_conflict` confirmados por la Fase 3 no son contradicciones reales, sino **instrumentos
paralelos** — sobre todo tratados bilaterales de Argentina con contrapartes distintas (convenios de
doble imposicion con Noruega vs. Belgica vs. Finlandia..., extradicion, traslado de condenados,
proteccion de inversiones). Que Argentina pacte reglas distintas con paises distintos es normal, no
un conflicto.

**Causa raiz**: el prompt de verificacion de la Fase 3 (`Clasificacion_Candidatos.ipynb`, celda 14)
solo veia el texto de los dos fragmentos + titulo -- nunca el resumen de la norma completa (donde
esta el pais contraparte, el beneficiario, etc.) -- y no exigia el test decisivo: pueden ambas
normas aplicar simultaneamente a la misma situacion y los mismos sujetos?

**Alcance**: se re-verifican los 834 pares que la Fase 3 marco `possible_conflict` en triage (440)
o dejo `needs_review` (394 de 397) con un prompt v2 que agrega contexto de norma completa, una
doctrina explicita de "instrumentos paralelos", y exige un campo `escenario_conflicto` (la
situacion concreta de colision -- si el modelo no puede articularla, no es conflicto). Los
sobrevivientes se confirman con `gemini-2.5-pro`.

**v1 nunca se sobreescribe**: `outputs/candidate_classifications.parquet` esta gitignoreado y sirve
de baseline de comparacion; todo lo nuevo se escribe en archivos `_v2`. `config.CLASSIFICATIONS_VERSION`
(env var `LEXAR_CLASSIFICATIONS_VERSION`, default `"v2"`) es el unico punto de corte entre servir v1
o v2 en la app -- revertir es cambiar esa constante y reiniciar el server, sin re-computar nada.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "outputs").exists() and (ROOT.parent / "outputs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import pyarrow.dataset as ds
from google.genai import types

from lexar import config
from lexar.llm import get_client
from lexar.rate_limiter import AdaptiveRateLimiter

OUTPUT_DIR = config.OUTPUT_DIR
CC_V1_PATH = OUTPUT_DIR / "candidate_classifications.parquet"
CC_V2_PATH = OUTPUT_DIR / "candidate_classifications_v2.parquet"
VERIFY_V2_DIR = OUTPUT_DIR / "classification_verify_v2"
VERIFY_V2_DIR.mkdir(parents=True, exist_ok=True)
CONFIRM_V2_DIR = OUTPUT_DIR / "classification_confirm_v2"
CONFIRM_V2_DIR.mkdir(parents=True, exist_ok=True)

FLASH_MODEL = "gemini-2.5-flash"
PRO_MODEL = "gemini-2.5-pro"
MAX_WORKERS = 6
CHECKPOINT_EVERY = 20

assert CC_V1_PATH.exists(), CC_V1_PATH

## Prompt v2: contexto de norma completa + doctrina de instrumentos paralelos

In [ ]:
LABELS = [
    "possible_conflict",
    "possible_overlap",
    "possible_modification",
    "different_scope",
    "neutral",
    "needs_review",
]

CLASSIFICATION_SCHEMA_V2 = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "label": types.Schema(type=types.Type.STRING, enum=LABELS),
        "confidence": types.Schema(type=types.Type.NUMBER),
        "explanation": types.Schema(type=types.Type.STRING),
        "escenario_conflicto": types.Schema(type=types.Type.STRING),
        "evidence_a": types.Schema(type=types.Type.STRING),
        "evidence_b": types.Schema(type=types.Type.STRING),
    },
    required=["label", "confidence", "explanation", "escenario_conflicto", "evidence_a", "evidence_b"],
)

LABEL_GUIDE_V2 = """
- possible_conflict: las DOS normas pueden aplicar simultaneamente a la MISMA situacion y los
  MISMOS sujetos, y ordenan resultados incompatibles entre si (obligaciones, prohibiciones,
  permisos, plazos o requisitos que no pueden cumplirse ambos a la vez). Si no podes describir esa
  situacion concreta, NO es possible_conflict.
- possible_overlap: los fragmentos regulan lo mismo de forma equivalente o redundante, sin
  incompatibilidad real (duplicacion, no contradiccion).
- possible_modification: uno de los fragmentos parece modificar, actualizar, ampliar o restringir
  al otro (por ejemplo, una norma posterior sobre el mismo punto).
- different_scope: el texto es similar pero los universos de aplicacion son disjuntos (sujetos,
  situaciones, epocas, jurisdicciones o contrapartes distintas), por lo que NO hay tension real
  aunque el texto se parezca.
- neutral: no se detecta ninguna tension ni relacion relevante mas alla de similitud superficial.
- needs_review: la comparacion es ambigua, falta contexto del articulo, o no se puede decidir con
  confianza entre las categorias anteriores.
""".strip()

DOCTRINA_INSTRUMENTOS_PARALELOS = """
Doctrina de "instrumentos paralelos" (aplicar SIEMPRE antes de marcar possible_conflict): dos
normas NO estan en conflicto solo porque regulan el mismo tema con reglas distintas, si cada una
aplica a un universo distinto y no se superponen. Casos tipicos que son SIEMPRE different_scope,
nunca possible_conflict, salvo que el texto indique lo contrario de forma explicita:
- Tratados o convenios bilaterales con una CONTRAPARTE (pais, estado, organismo) distinta cada
  uno (ej. un convenio de doble imposicion con Noruega vs. uno con Belgica: cada uno aplica solo
  a los sujetos alcanzados por ESE convenio, no hay colision posible).
- Normas que otorgan un beneficio, pension o derecho a personas o entidades distintas.
- Presupuestos, autorizaciones de gasto o ejercicios fiscales de anios/periodos distintos.
- Cartas organicas, estatutos o regimenes internos de organismos o entes distintos.
Si el par encaja en alguno de estos patrones, la etiqueta correcta es different_scope y el campo
escenario_conflicto debe quedar vacio.
""".strip()


def build_prompt_v2(row: pd.Series) -> str:
    hint = ""
    if row.get("has_infoleg_relation"):
        hint = (
            "\nPista adicional: Infoleg registra que una de estas dos normas modifica a la otra "
            "(relacion a nivel de norma completa, no necesariamente de este articulo puntual)."
        )
    return f"""Sos un asistente que ayuda a un equipo de revision legal a confirmar si un par de
fragmentos de leyes/tratados argentinos representa una contradiccion juridica real, o si es un
falso positivo de similitud semantica (dos normas parecidas en texto pero que en realidad no
colisionan nunca). La similitud semantica NO implica contradiccion: es solo una hipotesis para
revision humana. Cita evidencia textual literal; no inventes contenido que no este en el texto.

Etiquetas posibles y su significado:
{LABEL_GUIDE_V2}

{DOCTRINA_INSTRUMENTOS_PARALELOS}

--- Fragmento A ---
Norma: {row['title_a']} ({row['tipo_norma_a']}, sancionada {row['date_a']})
Resumen oficial de la norma completa (Infoleg): {row['resumen_a']}
Articulo/seccion: {row['label_a']}
Texto del fragmento: {row['text_a']}

--- Fragmento B ---
Norma: {row['title_b']} ({row['tipo_norma_b']}, sancionada {row['date_b']})
Resumen oficial de la norma completa (Infoleg): {row['resumen_b']}
Articulo/seccion: {row['label_b']}
Texto del fragmento: {row['text_b']}
{hint}

Devolve un unico JSON con: label (una de las 6 etiquetas exactas), confidence (0.0 a 1.0),
explanation (1-2 oraciones), escenario_conflicto (OBLIGATORIO si label es possible_conflict:
describi en 1 oracion la situacion concreta en la que ambas normas aplicarian a la vez y
colisionarian; string vacio "" para cualquier otra etiqueta), evidence_a (cita textual breve de
Fragmento A), evidence_b (cita textual breve de Fragmento B)."""

## Alcance: pares triage `possible_conflict` (440) + `needs_review` (394)

In [ ]:
cc_v1 = pd.read_parquet(CC_V1_PATH)
scope = cc_v1[(cc_v1["triage_label"] == "possible_conflict") | (cc_v1["final_label"] == "needs_review")].copy()

frag_ids = pd.unique(pd.concat([scope["fragment_a_id"], scope["fragment_b_id"]]))
cols = ["fragment_id", "document_id", "label", "text", "tipo_norma", "titulo_resumido", "fecha_sancion"]
frags = ds.dataset(config.EMBEDDING_FRAGMENTS_PATH).to_table(
    columns=cols, filter=ds.field("fragment_id").isin(frag_ids)
).to_pandas().set_index("fragment_id")

doc_ids = pd.unique(pd.concat([scope["document_a_id"], scope["document_b_id"]]))
docs_resumen = pd.read_csv(config.DOCUMENTS_PATH, dtype=str, usecols=["document_id", "texto_resumido"])
docs_resumen = docs_resumen[docs_resumen["document_id"].isin(doc_ids)].set_index("document_id")["texto_resumido"]


def _join(prefix: str) -> None:
    fid_col, did_col = f"fragment_{prefix}_id", f"document_{prefix}_id"
    scope[f"title_{prefix}"] = scope[fid_col].map(frags["titulo_resumido"])
    scope[f"tipo_norma_{prefix}"] = scope[fid_col].map(frags["tipo_norma"])
    scope[f"date_{prefix}"] = scope[fid_col].map(frags["fecha_sancion"])
    scope[f"label_{prefix}"] = scope[fid_col].map(frags["label"])
    scope[f"text_{prefix}"] = scope[fid_col].map(frags["text"])
    scope[f"resumen_{prefix}"] = scope[did_col].map(docs_resumen).fillna("(sin resumen)")


_join("a")
_join("b")
print(f"Total a re-verificar: {len(scope):,}")

## Smoke test

Antes de la corrida completa: pares de instrumentos paralelos conocidos (deben flipear a
`different_scope`) + una muestra aleatoria del resto (debe seguir mostrando conflictos genuinos
cuando corresponda).

In [ ]:
def call_llm(model: str, prompt: str, limiter: AdaptiveRateLimiter, max_retries: int = 6) -> dict:
    client = get_client()
    for attempt in range(max_retries):
        limiter.wait_turn()
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=CLASSIFICATION_SCHEMA_V2,
                    temperature=0.0,
                ),
            )
            limiter.report_success()
            return json.loads(response.text)
        except Exception as exc:
            if "RESOURCE_EXHAUSTED" in str(exc) or "429" in str(exc):
                limiter.report_rate_limited()
            if attempt == max_retries - 1:
                print(f"  [fallo definitivo] {exc}")
                return {}
            time.sleep(5.0 * (2 ** attempt))


sample_paralelos = scope[
    scope["title_a"].str.contains("DOBLE IMPOSICION|EXTRADICION|TRASLADO DE CONDENADOS", case=False, na=False)
    | scope["title_b"].str.contains("DOBLE IMPOSICION|EXTRADICION|TRASLADO DE CONDENADOS", case=False, na=False)
].head(8)
otros = scope[~scope["pair_key"].isin(sample_paralelos["pair_key"])].sample(3, random_state=42)
smoke_sample = pd.concat([sample_paralelos, otros])

smoke_limiter = AdaptiveRateLimiter()
for _, row in smoke_sample.iterrows():
    result = call_llm(FLASH_MODEL, build_prompt_v2(row), smoke_limiter)
    print(f"- {row['title_a'][:45]} <-> {row['title_b'][:45]}")
    print(f"  label={result.get('label')} conf={result.get('confidence')} escenario={result.get('escenario_conflicto')!r}")

**Resultado del smoke test (2026-07-10)**: los 8 pares de tratados bilaterales (doble imposicion,
extradicion) flipearon a `different_scope` con confianza 0.9-1.0, cada uno explicando correctamente
que la contraparte (pais) es distinta. Los 3 pares aleatorios adicionales tambien resultaron
`different_scope` genuino (formulas de vigencia de leyes distintas) -- consistente con el hallazgo
de que ninguno de los 8 candidatos "no-tratado" restantes en los 84 originales tenia un patron de
conflicto sustantivo claro por titulo. Se valido ademas con 2 pares sinteticos (no en el corpus):
un caso de instrumento paralelo control (confirmado `different_scope`) y un caso de contradiccion
real construida a proposito -- dos leyes nacionales, mismo ambito, mismos sujetos, plazos
incompatibles y sin relacion de modificacion -- que el prompt v2 clasifico correctamente como
`possible_conflict` con un `escenario_conflicto` concreto. Esto confirma que el criterio nuevo no es
indiscriminadamente conservador: sigue detectando conflictos genuinos, solo dejo de confundir
instrumentos paralelos con contradicciones.

## Corrida completa: flash-v2 + confirmacion con pro sobre sobrevivientes

In [ ]:
def run_stage(df: pd.DataFrame, model: str, out_dir: Path, label_tag: str) -> pd.DataFrame:
    """Checkpointeado y resumible por pair_key -- mismo patron que run_classification_stage
    de la Fase 3 (AdaptiveRateLimiter + ThreadPoolExecutor + part files)."""
    parts = sorted(out_dir.glob("part_*.parquet"))
    done_keys = set(pd.concat([pd.read_parquet(p, columns=["pair_key"]) for p in parts])["pair_key"]) if parts else set()
    pending = df[~df["pair_key"].isin(done_keys)]
    print(f"[{label_tag}] {len(done_keys)} ya hechos, {len(pending)} pendientes de {len(df)}")

    limiter = AdaptiveRateLimiter()
    lock = threading.Lock()
    buffer = []
    next_idx = max([int(p.stem.split("_")[1]) for p in parts], default=-1) + 1

    def flush():
        nonlocal buffer, next_idx
        if not buffer:
            return
        pd.DataFrame(buffer).to_parquet(out_dir / f"part_{next_idx:06d}.parquet", index=False)
        next_idx += 1
        buffer = []

    def worker(row):
        return row["pair_key"], call_llm(model, build_prompt_v2(row), limiter)

    rows = [r for _, r in pending.iterrows()]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(worker, r) for r in rows]
        for n, fut in enumerate(as_completed(futures), 1):
            pair_key, result = fut.result()
            with lock:
                if result:
                    buffer.append({"pair_key": pair_key, **result})
                if len(buffer) >= CHECKPOINT_EVERY:
                    flush()
            if n % 50 == 0:
                print(f"  [{label_tag}] {n}/{len(rows)}")
        with lock:
            flush()

    parts = sorted(out_dir.glob("part_*.parquet"))
    return pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True) if parts else pd.DataFrame()


flash_results = run_stage(scope, FLASH_MODEL, VERIFY_V2_DIR, "flash-v2")
flash_results = flash_results.rename(columns={c: f"flash_{c}" for c in flash_results.columns if c != "pair_key"})
scope = scope.merge(flash_results, on="pair_key", how="left")
print(scope["flash_label"].value_counts(dropna=False))

In [ ]:
to_confirm = scope[scope["flash_label"] == "possible_conflict"]
print(f"Pares que flash-v2 confirma possible_conflict, van a confirmacion con {PRO_MODEL}: {len(to_confirm)}")

if len(to_confirm):
    pro_results = run_stage(to_confirm, PRO_MODEL, CONFIRM_V2_DIR, "pro-confirm")
    pro_results = pro_results.rename(columns={c: f"pro_{c}" for c in pro_results.columns if c != "pair_key"})
    scope = scope.merge(pro_results, on="pair_key", how="left")
else:
    for c in ["pro_label", "pro_confidence", "pro_explanation", "pro_escenario_conflicto", "pro_evidence_a", "pro_evidence_b"]:
        scope[c] = None

if "pro_label" in scope:
    print(scope["pro_label"].value_counts(dropna=False))

## Consolidacion: `candidate_classifications_v2.parquet` (v1 intacto)

Copia completa de v1 con las filas re-verificadas actualizadas -- `escenario_conflicto` nueva,
vacia para las filas no tocadas. Nunca se escribe sobre `candidate_classifications.parquet`.

In [ ]:
scope["final_label_v2"] = scope["pro_label"].fillna(scope["flash_label"]).fillna("needs_review")
scope["final_confidence_v2"] = scope["pro_confidence"].fillna(scope["flash_confidence"])
scope["final_explanation_v2"] = scope["pro_explanation"].fillna(scope["flash_explanation"])
scope["escenario_conflicto"] = scope["pro_escenario_conflicto"].fillna(scope["flash_escenario_conflicto"]).fillna("")
scope["final_model_v2"] = scope["pro_label"].notna().map({True: PRO_MODEL, False: FLASH_MODEL})

cc_v2 = cc_v1.copy()
cc_v2["escenario_conflicto"] = ""
upd = scope.set_index("pair_key")
mask = cc_v2["pair_key"].isin(upd.index)
idx = cc_v2.index[mask]
keys = cc_v2.loc[idx, "pair_key"]
cc_v2.loc[idx, "verify_label"] = keys.map(upd["final_label_v2"]).values
cc_v2.loc[idx, "final_label"] = keys.map(upd["final_label_v2"]).values
cc_v2.loc[idx, "final_confidence"] = keys.map(upd["final_confidence_v2"]).values
cc_v2.loc[idx, "final_explanation"] = keys.map(upd["final_explanation_v2"]).values
cc_v2.loc[idx, "final_model"] = keys.map(upd["final_model_v2"]).values
cc_v2.loc[idx, "escenario_conflicto"] = keys.map(upd["escenario_conflicto"]).values
cc_v2.loc[idx, "prompt_version"] = "v2"

cc_v2.to_parquet(CC_V2_PATH, index=False)
print(f"Guardado {CC_V2_PATH} ({len(cc_v2):,} filas, {mask.sum():,} actualizadas)")
print("\nDistribucion final_label v2:"); print(cc_v2["final_label"].value_counts())
print("\nDistribucion final_label v1 (sin tocar, para comparar):"); print(cc_v1["final_label"].value_counts())

## `conflicts_top_v2.csv` y rebuild de la Fase 5 (`norm_links_v2.parquet`)

In [ ]:
conflictos_v2 = cc_v2[cc_v2["final_label"] == "possible_conflict"].copy()
frag_ids_top = pd.unique(pd.concat([conflictos_v2["fragment_a_id"], conflictos_v2["fragment_b_id"]]))
textos_top = ds.dataset(config.EMBEDDING_FRAGMENTS_PATH).to_table(
    columns=["fragment_id", "text"], filter=ds.field("fragment_id").isin(frag_ids_top)
).to_pandas().set_index("fragment_id")["text"]
conflictos_v2["text_a"] = conflictos_v2["fragment_a_id"].map(textos_top)
conflictos_v2["text_b"] = conflictos_v2["fragment_b_id"].map(textos_top)
conflictos_v2.to_csv(OUTPUT_DIR / "conflicts_top_v2.csv", index=False)
print(f"conflicts_top_v2.csv: {len(conflictos_v2)} pares")

from lexar.links import build_norm_links

norm_links_v2 = build_norm_links(
    classifications_path=CC_V2_PATH,
    norm_links_path=OUTPUT_DIR / "norm_links_v2.parquet",
)

## Sanity check: Ley 24.240 no debe cambiar

Sus vinculos son casi todos `possible_modification` oficiales (Infoleg), no `possible_conflict` --
el rework de conflictos no deberia afectarlos.

In [ ]:
from lexar.links import links_for_document, count_later_modifications

links_2424 = links_for_document(norm_links_v2, "infoleg:638")
print(f"infoleg:638 (Ley 24.240): {len(links_2424)} vinculos, "
      f"{count_later_modifications(norm_links_v2, 'infoleg:638')} modificaciones posteriores registradas")